In [ ]:
import os, sys, json
from datetime import datetime
import math
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()
MODEL = DEFAULT_MODEL


## 1. Definir Tools (Schemas)

In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()
    MODEL = DEFAULT_MODEL

# Definición de herramientas en formato JSON Schema
tools = [
    {
        "name": "calculadora",
        "description": "Realiza operaciones matemáticas: suma, resta, multiplicación, división, potencia, raíz cuadrada.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operacion": {
                    "type": "string",
                    "enum": ["suma", "resta", "multiplicacion", "division", "potencia", "raiz"],
                    "description": "La operación matemática a realizar"
                },
                "a": {"type": "number", "description": "Primer operando"},
                "b": {"type": "number", "description": "Segundo operando (no requerido para raíz)"}
            },
            "required": ["operacion", "a"]
        }
    },
    {
        "name": "obtener_tiempo",
        "description": "Obtiene la fecha y hora actual del sistema.",
        "input_schema": {
            "type": "object",
            "properties": {
                "zona_horaria": {
                    "type": "string",
                    "description": "Zona horaria (ej: 'America/Bogota', 'Europe/Madrid')",
                    "default": "UTC"
                }
            }
        }
    }
]

print(f"OK: {len(tools)} herramientas definidas: {[t['name'] for t in tools]}")


## 2. Implementar las Funciones

In [ ]:
def ejecutar_calculadora(operacion: str, a: float, b: float = None) -> dict:
    """Ejecuta la operación matemática solicitada."""
    try:
        if operacion == "suma":          resultado = a + b
        elif operacion == "resta":       resultado = a - b
        elif operacion == "multiplicacion": resultado = a * b
        elif operacion == "division":
            if b == 0: return {"error": "División por cero"}
            resultado = a / b
        elif operacion == "potencia":    resultado = a ** b
        elif operacion == "raiz":        resultado = math.sqrt(a)
        else: return {"error": f"Operación desconocida: {operacion}"}
        return {"resultado": resultado}
    except Exception as e:
        return {"error": str(e)}

def ejecutar_obtener_tiempo(zona_horaria: str = "UTC") -> dict:
    """Retorna la fecha y hora actual."""
    ahora = datetime.now()
    return {
        "fecha": ahora.strftime("%Y-%m-%d"),
        "hora": ahora.strftime("%H:%M:%S"),
        "dia_semana": ahora.strftime("%A"),
        "zona": zona_horaria
    }

def ejecutar_herramienta(nombre: str, inputs: dict) -> str:
    """Dispatcher: ejecuta la herramienta correspondiente."""
    if nombre == "calculadora":
        result = ejecutar_calculadora(**inputs)
    elif nombre == "obtener_tiempo":
        result = ejecutar_obtener_tiempo(**inputs)
    else:
        result = {"error": f"Herramienta '{nombre}' no encontrada"}
    return json.dumps(result)

# Prueba directa
print(ejecutar_calculadora("suma", 15, 27))
print(ejecutar_obtener_tiempo())

## 3. Bucle de Tool Use

In [ ]:
def chat_with_tools(user_message: str, tools: list, verbose: bool = True) -> str:
    """Implementa el bucle completo de tool use."""
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        if verbose:
            print(f"  stop_reason: {response.stop_reason}")
        
        # Si Claude terminó, retornar respuesta final
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if hasattr(b, 'text')]
            return " ".join(text_blocks)
        
        # Si Claude quiere usar herramientas
        if response.stop_reason == "tool_use":
            # Añadir respuesta de Claude al historial
            messages.append({"role": "assistant", "content": response.content})
            
            # Ejecutar cada tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  🔧 Usando: {block.name}({block.input})")
                    
                    result = ejecutar_herramienta(block.name, block.input)
                    
                    if verbose:
                        print(f"  ✅ Resultado: {result}")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            # Añadir resultados al historial
            messages.append({"role": "user", "content": tool_results})
        else:
            break
    
    return "Error: respuesta inesperada"

# Prueba
print("Usuario: ¿Cuánto es 234 * 567?")
respuesta = chat_with_tools("¿Cuánto es 234 multiplicado por 567?", tools)
print(f"Claude: {respuesta}")

In [ ]:
# Prueba con múltiples herramientas en una sola pregunta
print("=== Múltiples herramientas ===")
respuesta = chat_with_tools(
    "¿Qué hora es ahora? Y además, ¿cuál es la raíz cuadrada de 144?",
    tools
)
print(f"Claude: {respuesta}")

## 4. Tool Use con Herramienta de Búsqueda Web (Simulada)

In [ ]:
# Herramienta de búsqueda web simulada
web_search_tool = {
    "name": "buscar_web",
    "description": "Busca información actualizada en internet sobre cualquier tema.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "La consulta de búsqueda"
            },
            "num_resultados": {
                "type": "integer",
                "description": "Número de resultados a retornar (1-5)",
                "default": 3
            }
        },
        "required": ["query"]
    }
}

# Base de datos simulada para la búsqueda
KNOWLEDGE_BASE = {
    "python": "Python es un lenguaje de programación interpretado, de alto nivel. Versión actual: 3.12",
    "claude": "Claude es un asistente de IA desarrollado por Anthropic. Modelo actual: Claude Sonnet 4.5",
    "inteligencia artificial": "La IA es la simulación de procesos de inteligencia humana por sistemas informáticos.",
}

def ejecutar_buscar_web(query: str, num_resultados: int = 3) -> str:
    """Simula una búsqueda web."""
    query_lower = query.lower()
    resultados = []
    for key, value in KNOWLEDGE_BASE.items():
        if any(word in query_lower for word in key.split()):
            resultados.append({"titulo": key.title(), "resumen": value})
    
    if not resultados:
        return json.dumps({"resultados": [], "mensaje": "No se encontraron resultados"})
    
    return json.dumps({"resultados": resultados[:num_resultados]})

# Registrar la nueva herramienta en el dispatcher
def ejecutar_herramienta_v2(nombre: str, inputs: dict) -> str:
    if nombre == "calculadora":    return json.dumps(ejecutar_calculadora(**inputs))
    elif nombre == "obtener_tiempo": return json.dumps(ejecutar_obtener_tiempo(**inputs))
    elif nombre == "buscar_web":   return ejecutar_buscar_web(**inputs)
    return json.dumps({"error": f"Herramienta '{nombre}' no encontrada"})

all_tools = tools + [web_search_tool]

# Sobreescribir dispatcher en el bucle
def chat_with_tools_v2(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=all_tools, messages=messages
        )
        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, 'text'))
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    res = ejecutar_herramienta_v2(block.name, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": res})
            messages.append({"role": "user", "content": results})
        else:
            break
    return ""

print("=== Búsqueda web + cálculo ===")
resp = chat_with_tools_v2("Busca información sobre Python y también calcula 2 elevado a la potencia 10")
print(f"\nClaude: {resp}")

## 5. Fine-Grained Tool Calling: tool_choice

In [ ]:
# tool_choice permite controlar qué herramientas usa Claude:
# - {"type": "auto"}   → Claude decide (default)
# - {"type": "any"}    → Claude DEBE usar al menos una herramienta
# - {"type": "tool", "name": "X"} → Claude DEBE usar esa herramienta específica

# Forzar uso de herramienta específica
response = client.messages.create(
    model=MODEL, max_tokens=300,
    tools=tools,
    tool_choice={"type": "tool", "name": "obtener_tiempo"},
    messages=[{"role": "user", "content": "Hola, buenos días"}]
)

print("Claude fue forzado a usar 'obtener_tiempo':")
for block in response.content:
    if block.type == "tool_use":
        print(f"  Herramienta: {block.name}")
        print(f"  Input: {block.input}")